In [1]:
!pip install -q youtube-transcript-api langchain-community faiss-cpu tiktoken python-dotenv langchain-text-splitters


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from youtube_transcript_api.proxies import WebshareProxyConfig
from youtube_transcript_api._errors import TranscriptsDisabled, NoTranscriptFound
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

load_dotenv()

C:\Users\aviru\AppData\Local\Temp\ipykernel_18796\2594908203.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


True

In [3]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001"
)

In [9]:
video_id = "LPZh9BOjkQs"

try:
    # If you don't care which language, this returns the "best" one
    transcript_list = YouTubeTranscriptApi().fetch(video_id, languages=["en"]).to_raw_data()

    # Flatten it to plain text
    transcript = " ".join(chunk["text"] for chunk in transcript_list)
    print(transcript)

except TranscriptsDisabled:
    print("No captions available for this video.")
except NoTranscriptFound:
    print("No English transcript found for this video.")

[Submit subtitle corrections at criblate.com]
Imagine you happen across a short movie script that describes a scene between a person and their AI assistant. The script has what the person asks the AI, but the AI's response has been torn off. Suppose you also have this powerful magical machine that can take any text and provide a sensible prediction of what word comes next. You could then finish the script by feeding in what you have to the machine, seeing what it would predict to start the AI's answer, and then repeating this over and over with a growing script completing the dialogue. When you interact with a chatbot, this is exactly what's happening. A large language model is a sophisticated mathematical function that predicts what word comes next for any piece of text. Instead of predicting one word with certainty, though, what it does is assign a probability to all possible next words. To build a chatbot, what you do is lay out some text that describes an interaction between a user

In [10]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])

In [16]:
chunks[9]

Document(metadata={}, page_content="probability for every possible next word. Although researchers design the framework for how each of these steps work, it's important to understand that the specific behavior is an emergent phenomenon based on how those hundreds of billions of parameters are tuned during training. This makes it incredibly challenging to determine why the model makes the exact predictions that it does. What you can see is that when you use large language model predictions to autocomplete a prompt, the words that it generates are uncannily fluent, fascinating, and even useful. If you're a new viewer and you're curious about more details on how transformers and attention work, boy do I have some material for you. One option is to jump into a series I made about deep learning, where we visualize and motivate the details of attention and all the other steps in a transformer. Also, on my second channel I just posted a talk I gave a couple months ago about this topic for the

In [13]:
vector_store = FAISS.from_documents(chunks, embeddings)

In [14]:
vector_store.index_to_docstore_id

{0: '16f83796-9926-404f-a396-26822b8a42a1',
 1: 'dbbec362-0410-4a97-9a2c-8425fa354011',
 2: 'a27cc9de-1bc0-4b27-8624-2a14950945c2',
 3: 'ce613836-70e2-44f7-a997-408102dded9a',
 4: 'b5d9330e-6632-491c-ac57-d465fa7edec6',
 5: '507b5a76-efbc-48d4-bce4-5d63e4f38c57',
 6: '855f5d59-9aeb-4ca2-ab25-c1a7073ae38b',
 7: '0caa8817-d849-465f-8bf7-f9a37181606a',
 8: 'ddc3aca5-f3bf-46f1-a379-1f1fb6924c37',
 9: 'de331d69-ed55-4eb8-a7ce-6cdb25585166',
 10: '36740db8-f98b-4d2f-a568-d5161809e3c9'}

In [19]:
vector_store.get_by_ids(['dbbec362-0410-4a97-9a2c-8425fa354011'])

[Document(id='dbbec362-0410-4a97-9a2c-8425fa354011', metadata={}, page_content="Imagine you happen across a short movie script that describes a scene between a person and their AI assistant. The script has what the person asks the AI, but the AI's response has been torn off. Suppose you also have this powerful magical machine that can take any text and provide a sensible prediction of what word comes next. You could then finish the script by feeding in what you have to the machine, seeing what it would predict to start the AI's answer, and then repeating this over and over with a growing script completing the dialogue. When you interact with a chatbot, this is exactly what's happening. A large language model is a sophisticated mathematical function that predicts what word comes next for any piece of text. Instead of predicting one word with certainty, though, what it does is assign a probability to all possible next words. To build a chatbot, what you do is lay out some text that descr

In [20]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [22]:

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)

In [23]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [24]:
question = "How does a chatbot generate its response, according to the text?"
retrieved_docs = retriever.invoke(question)

In [25]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

"does is assign a probability to all possible next words. To build a chatbot, what you do is lay out some text that describes an interaction between a user and a hypothetical AI assistant, you add on whatever the user types in as the first part of the interaction, and then have the model repeatedly predict the next word that such a hypothetical AI assistant would say in response, and that's what's presented to the user. In doing this, the output tends to look a lot more natural if you allow it to select less likely words along the way at random. So what this means is even though the model itself is deterministic, a given prompt typically gives a different answer each time it's run. Models learn how to make these predictions by processing an enormous amount of text, typically pulled from the internet. For a standard human to read the amount of text that was used to train GPT-3, for example, if they read non-stop 24-7, it would take over 2600 years. Larger models since then train on\n\nI

In [26]:
final_prompt = prompt.invoke({"context": context_text, "question": question})
final_prompt

StringPromptValue(text="\n      You are a helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say you don't know.\n\n      does is assign a probability to all possible next words. To build a chatbot, what you do is lay out some text that describes an interaction between a user and a hypothetical AI assistant, you add on whatever the user types in as the first part of the interaction, and then have the model repeatedly predict the next word that such a hypothetical AI assistant would say in response, and that's what's presented to the user. In doing this, the output tends to look a lot more natural if you allow it to select less likely words along the way at random. So what this means is even though the model itself is deterministic, a given prompt typically gives a different answer each time it's run. Models learn how to make these predictions by processing an enormous amount of text, typically pulled from the internet

In [27]:
answer = llm.invoke(final_prompt)
print(answer.content)

According to the text, a chatbot generates its response by:

1. **Starting with a prompt** that describes the interaction between a user and a hypothetical AI assistant.  
2. **Using a large language model** that has been pre‑trained on massive amounts of text.  
3. **Predicting the next word**: the model assigns a probability to every possible next word.  
4. **Sampling from that probability distribution** (often allowing less‑likely words to be chosen at random) to pick the next word.  
5. **Repeating the process**: the chosen word is appended to the prompt, and the model again predicts the next word, continuing until the response is complete.  

This iterative next‑word prediction, guided by the model’s learned probabilities, produces the chatbot’s reply.
